# Dogs vs Cats — ResNet18 전이학습 (PyTorch) — Colab

Kaggle [Dogs vs Cats Redux](https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition) (개/고양이 이미지 분류) 를 **사전학습된 ResNet18 미세조정(transfer learning)** 으로 푸는 기본 예제입니다.

- ImageNet 으로 사전학습된 ResNet18 을 불러와 마지막 분류층만 2클래스(cat/dog)로 교체해 미세조정합니다.
- 실행 시 **Kaggle API로 데이터를 직접 다운로드**하며, 토큰이 **노트북에 내장**되어 바로 실행됩니다.
- 마지막에 제출 파일 `submission.csv` (`id,label` = 개일 확률) 가 `/content` 에 생성됩니다.

> ⚙️ **GPU 권장**: 런타임 → 런타임 유형 변경 → GPU (이미지 25,000장이라 CPU는 매우 느림).  
> ⚠️ **사전 필수**: `jinusun` 계정으로 [대회 규칙](https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition/rules)에 **Join/Accept** 필요 (안 하면 403).  
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "torchvision", "numpy", "pandas", "pillow"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Dogs vs Cats 데이터 다운로드

> 데이터가 약 800MB라 다운로드/압축 해제에 수 분 걸릴 수 있습니다.

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()

api.competition_download_files("dogs-vs-cats-redux-kernels-edition", path=WORK_DIR, quiet=False)
# 바깥 zip 해제
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
# train.zip / test.zip 해제
for fn, out in [("train.zip", "."), ("test.zip", ".")]:
    zp = os.path.join(WORK_DIR, fn)
    if os.path.exists(zp):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)

TRAIN_DIR = os.path.join(WORK_DIR, "train")
TEST_DIR = os.path.join(WORK_DIR, "test")
print("train images:", len(os.listdir(TRAIN_DIR)), "| test images:", len(os.listdir(TEST_DIR)))

## 2. Dataset / DataLoader

파일명으로 라벨을 정합니다 (`cat.123.jpg` → 0, `dog.123.jpg` → 1). ImageNet 사전학습 모델에 맞춰 224×224 리사이즈 + 정규화하고, 학습셋엔 좌우반전 증강을 적용합니다.

In [ ]:
import glob
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class DogsCats(Dataset):
    def __init__(self, files, tf, with_label=True):
        self.files = files; self.tf = tf; self.with_label = with_label
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        path = self.files[i]
        img = self.tf(Image.open(path).convert("RGB"))
        if self.with_label:
            label = 1 if "dog" in os.path.basename(path) else 0
            return img, label
        # 테스트: 파일명 숫자 id 반환
        img_id = int(os.path.splitext(os.path.basename(path))[0])
        return img, img_id

all_train = sorted(glob.glob(os.path.join(TRAIN_DIR, "*.jpg")))
import numpy as np
rng = np.random.RandomState(42); rng.shuffle(all_train)
cut = int(len(all_train) * 0.9)
train_files, val_files = all_train[:cut], all_train[cut:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NW = 2
train_loader = DataLoader(DogsCats(train_files, train_tf), batch_size=64, shuffle=True, num_workers=NW)
val_loader = DataLoader(DogsCats(val_files, eval_tf), batch_size=64, shuffle=False, num_workers=NW)
print("device:", device, "| train:", len(train_files), "val:", len(val_files))

## 3. ResNet18 불러오기 (사전학습) & 분류층 교체

In [ ]:
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
# 마지막 완전연결층을 2클래스로 교체
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)
print("ResNet18 준비 완료 | fc:", model.fc)

## 4. 미세조정 학습

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 3
for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimizer.step()
    # 검증 정확도
    model.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            pred = model(xb).argmax(1).cpu()
            correct += (pred == yb).sum().item(); total += len(yb)
    print(f"epoch {epoch} | val accuracy = {correct/total:.4f}")

## 5. 테스트 예측 & 제출 파일 생성

제출은 '개일 확률'(softmax) 입니다.

In [ ]:
import pandas as pd
test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.jpg")))
test_loader = DataLoader(DogsCats(test_files, eval_tf, with_label=False),
                         batch_size=64, shuffle=False, num_workers=NW)

model.eval()
ids, probs = [], []
with torch.no_grad():
    for xb, img_ids in test_loader:
        p = torch.softmax(model(xb.to(device)), dim=1)[:, 1].cpu().numpy()  # P(dog)
        probs.extend(p.tolist()); ids.extend(img_ids.tolist())

submission_path = os.path.join(WORK_DIR, "submission.csv")
sub = pd.DataFrame({"id": ids, "label": np.clip(probs, 1e-4, 1 - 1e-4)}).sort_values("id")
sub.to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(sub))
sub.head()

## 6. 제출 파일 내려받기

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[Dogs vs Cats Redux 제출 페이지](https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition/submit)**

- 대회 홈: https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.